#Google Places API - Initial exploration

## Objective
Test a single API request and inspect the returned data before
building the ingestion pipeline.

## Initial sample
- Category: cafés
- Center: Plaza Abaroa, La Paz
- Radius: 600 meters
- Maximum results: 5

## Checks
- Response status and number of places returned
- Missing ratings and review counts
- Repeated place IDs
- Original place categories

## Status
A request was successfully tested in Postman.
The Python extraction has not been implemented yet.

In [0]:
# Disable Google API calls unless explicitly enabled for a manual extraction.

ENABLE_API_CALLS = False

In [0]:
#Defning the list for the API secret
# List secret scopes available in this workspace.
scopes = dbutils.secrets.listScopes()

for scope in scopes:
    print(scope.name)

if not scopes:
    print("No secret scopes found.")

In [0]:
#Saving the API key

from getpass import getpass
import warnings
from databricks.sdk import WorkspaceClient

# Use the exact scope name returned by listScopes().
scope_name = "places-explorer"

client = WorkspaceClient()

# Stop if hidden input is unavailable.
with warnings.catch_warnings():
    warnings.simplefilter("error", category=UserWarning)
    api_key = getpass("Google Places API key: ").strip()

try:
    if not api_key:
        raise ValueError("The API key cannot be empty.")

    client.secrets.put_secret(
        scope=scope_name,
        key="google-places-api-key",
        string_value=api_key
    )
finally:
    del api_key

print("API key saved.")

In [0]:
#Verifyng the API key

google_places_api_key = dbutils.secrets.get(
    scope="places-explorer",
    key="google-places-api-key"
)

print("API key loaded successfully.")

In [0]:
#Testing the API

import requests
from datetime import datetime, timezone

url = "https://places.googleapis.com/v1/places:searchNearby"

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": google_places_api_key,
    "X-Goog-FieldMask": (
    "places.id,places.displayName,places.primaryType,places.types,"
    "places.formattedAddress,places.location,places.businessStatus,"
    "places.rating,places.userRatingCount"
)
}

payload = {
    "includedTypes": ["cafe"],
    "maxResultCount": 5,
    "rankPreference": "DISTANCE",
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": -16.5108331,
                "longitude": -68.1265597
            },
            "radius": 600.0
        }
    }
}

extracted_at = datetime.now(timezone.utc).isoformat()

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=30
)

print(f"HTTP status: {response.status_code}")

if response.ok:
    response_data = response.json()
    places = response_data.get("places", [])

    print(f"Places returned: {len(places)}")
    print(f"Extraction timestamp (UTC): {extracted_at}")
else:
    print("Request failed. Check the HTTP status before continuing.")

In [0]:
#Verifying the response is saved locally

for place in places:
    print({
        "name": place.get("displayName", {}).get("text"),
        "primary_type": place.get("primaryType"),
        "address": place.get("formattedAddress"),
        "business_status": place.get("businessStatus"),
        "rating": place.get("rating"),
        "review_count": place.get("userRatingCount")
    })

In [0]:
import requests
from datetime import datetime, timezone


# Create a reusable function without an explicit result limit.
def fetch_nearby_places(
    api_key,
    included_types,
    latitude,
    longitude,
    radius_meters=600
):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": (
            "places.id,places.displayName,places.primaryType,places.types,"
            "places.formattedAddress,places.location,places.businessStatus,"
            "places.rating,places.userRatingCount"
        )
    }

    payload = {
        "includedTypes": included_types,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": latitude,
                    "longitude": longitude
                },
                "radius": radius_meters
            }
        }
    }

    extracted_at = datetime.now(timezone.utc).isoformat()

    response = requests.post(
        "https://places.googleapis.com/v1/places:searchNearby",
        headers=headers,
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    return {
        "extracted_at_utc": extracted_at,
        "request_parameters": payload,
        "response_data": response.json()
    }

In [0]:
#testing the function
result = fetch_nearby_places(
    api_key=google_places_api_key,
    included_types=["cafe"],
    latitude=-16.5108331,
    longitude=-68.1265597,
    radius_meters=600
)

places = result["response_data"].get("places", [])

print(f"Places returned: {len(places)}")
print(
    "maxResultCount sent:",
    "maxResultCount" in result["request_parameters"]
)

In [0]:
#adding logic to map the zones
import math

study_center = {
    "latitude": -16.5108331,
    "longitude": -68.1265597
}

spacing_meters = 300
search_radius_meters = 350
earth_radius_meters = 6_371_000

search_points = []

for north_offset in [-spacing_meters, 0, spacing_meters]:
    for east_offset in [-spacing_meters, 0, spacing_meters]:

        latitude = study_center["latitude"] + math.degrees(
            north_offset / earth_radius_meters
        )

        longitude = study_center["longitude"] + math.degrees(
            east_offset / (
                earth_radius_meters
                * math.cos(math.radians(study_center["latitude"]))
            )
        )

        search_points.append({
            "point_id": f"sopocachi_{len(search_points) + 1:02d}",
            "latitude": latitude,
            "longitude": longitude,
            "north_offset_meters": north_offset,
            "east_offset_meters": east_offset,
            "radius_meters": search_radius_meters
        })

for point in search_points:
    print(point)

print(f"Search points generated: {len(search_points)}")

In [0]:
#Testing mapping logic
# Select the first search point.
point = search_points[0]

# Make one Google Places request.
point_result = fetch_nearby_places(
    api_key=google_places_api_key,
    included_types=["cafe"],
    latitude=point["latitude"],
    longitude=point["longitude"],
    radius_meters=point["radius_meters"]
)

# Attach our own search metadata.
point_result["search_context"] = {
    "study_area": "sopocachi",
    "requested_category": "cafes",
    "point_id": point["point_id"],
    "study_center": study_center,
    "study_radius_meters": 600
}

point_places = point_result["response_data"].get("places", [])

print(f"Search point: {point['point_id']}")
print(f"Places returned: {len(point_places)}")
print(f"Potentially truncated: {len(point_places) == 20}")

In [0]:
#Still getting potential incomplete results, analyzing results
original_places = result["response_data"].get("places", [])

original_ids = {
    place["id"]
    for place in original_places
    if place.get("id")
}

point_ids = {
    place["id"]
    for place in point_places
    if place.get("id")
}

print(f"Original search — unique places: {len(original_ids)}")
print(f"Grid point — unique places: {len(point_ids)}")
print(f"Shared places: {len(original_ids & point_ids)}")
print(f"New places from grid point: {len(point_ids - original_ids)}")
print(f"Combined unique places: {len(original_ids | point_ids)}")

In [0]:
#testing fixed distance
def distance_meters(lat1, lon1, lat2, lon2):
    earth_radius = 6_371_000

    lat1, lon1, lat2, lon2 = map(
        math.radians, [lat1, lon1, lat2, lon2]
    )

    a = (
        math.sin((lat2 - lat1) / 2) ** 2
        + math.cos(lat1)
        * math.cos(lat2)
        * math.sin((lon2 - lon1) / 2) ** 2
    )

    return 2 * earth_radius * math.asin(math.sqrt(min(1, a)))


unique_places = {
    place["id"]: place
    for place in original_places + point_places
    if place.get("id")
}

coverage_counts = {
    "inside_study_area": 0,
    "outside_study_area": 0,
    "missing_location": 0
}

for place in unique_places.values():
    location = place.get("location", {})
    latitude = location.get("latitude")
    longitude = location.get("longitude")

    if latitude is None or longitude is None:
        coverage_counts["missing_location"] += 1
        continue

    distance = distance_meters(
        study_center["latitude"],
        study_center["longitude"],
        latitude,
        longitude
    )

    if distance <= 600:
        coverage_counts["inside_study_area"] += 1
    else:
        coverage_counts["outside_study_area"] += 1

print(coverage_counts)

In [0]:
#adding completeness logic 

search_results = [
    {
        "query_id": "sopocachi_cafes_initial",
        "study_area": "sopocachi",
        "requested_category": "cafes",
        "point_id": "initial_center",
        **result
    },
    {
        "query_id": f"sopocachi_cafes_{point['point_id']}",
        "study_area": "sopocachi",
        "requested_category": "cafes",
        "point_id": point["point_id"],
        **point_result
    }
]

for search_result in search_results:
    returned_places = search_result["response_data"].get("places", [])

    search_result["returned_count"] = len(returned_places)
    search_result["potentially_truncated"] = len(returned_places) == 20

print(f"Search responses registered: {len(search_results)}")

In [0]:
#testing rest of points

import time

completed_point_ids = {
    item["point_id"]
    for item in search_results
}

pending_points = [
    point
    for point in search_points
    if point["point_id"] not in completed_point_ids
]

max_calls = 8
points_to_query = pending_points[:max_calls]

for index, search_point in enumerate(points_to_query):
    # Space requests apart; no automatic retries.
    if index > 0:
        time.sleep(12)

    try:
        search_result = fetch_nearby_places(
            api_key=google_places_api_key,
            included_types=["cafe"],
            latitude=search_point["latitude"],
            longitude=search_point["longitude"],
            radius_meters=search_point["radius_meters"]
        )
    except requests.RequestException as error:
        print(
            f"Stopped at {search_point['point_id']}. "
            f"Error type: {type(error).__name__}"
        )
        break

    returned_count = len(
        search_result["response_data"].get("places", [])
    )

    search_result.update({
        "query_id": f"sopocachi_cafes_{search_point['point_id']}",
        "study_area": "sopocachi",
        "requested_category": "cafes",
        "point_id": search_point["point_id"],
        "returned_count": returned_count,
        "potentially_truncated": returned_count == 20
    })

    search_results.append(search_result)

    print(
        f"{search_point['point_id']} | "
        f"Returned: {returned_count} | "
        f"Potentially truncated: {returned_count == 20}"
    )

print(f"Total responses registered: {len(search_results)}")

In [0]:
# Summarize unique places, repeated appearances, and geographic coverage.

places_by_id = {}
total_appearances = 0
missing_id_count = 0

for search_result in search_results:
    returned_places = search_result["response_data"].get("places", [])
    total_appearances += len(returned_places)

    for place in returned_places:
        place_id = place.get("id")

        if not place_id:
            missing_id_count += 1
            continue

        if place_id not in places_by_id:
            places_by_id[place_id] = place

coverage_summary = {
    "responses": len(search_results),
    "total_appearances": total_appearances,
    "unique_places": len(places_by_id),
    "repeated_appearances": (
        total_appearances - missing_id_count - len(places_by_id)
    ),
    "appearances_without_id": missing_id_count,
    "inside_study_area": 0,
    "outside_study_area": 0,
    "missing_location": 0,
    "potentially_truncated_responses": sum(
        item["potentially_truncated"]
        for item in search_results
    )
}

for place in places_by_id.values():
    location = place.get("location") or {}
    latitude = location.get("latitude")
    longitude = location.get("longitude")

    if latitude is None or longitude is None:
        coverage_summary["missing_location"] += 1
        continue

    distance = distance_meters(
        study_center["latitude"],
        study_center["longitude"],
        latitude,
        longitude
    )

    if distance <= 600:
        coverage_summary["inside_study_area"] += 1
    else:
        coverage_summary["outside_study_area"] += 1

for metric, value in coverage_summary.items():
    print(f"{metric}: {value}")

In [0]:
# Measure each query's contribution of new places inside the study area.

seen_inside_ids = set()
query_contributions = []

for search_result in search_results:
    inside_ids = set()

    for place in search_result["response_data"].get("places", []):
        place_id = place.get("id")
        location = place.get("location") or {}
        latitude = location.get("latitude")
        longitude = location.get("longitude")

        if not place_id or latitude is None or longitude is None:
            continue

        distance = distance_meters(
            study_center["latitude"],
            study_center["longitude"],
            latitude,
            longitude
        )

        if distance <= 600:
            inside_ids.add(place_id)

    new_ids = inside_ids - seen_inside_ids
    seen_inside_ids.update(inside_ids)

    query_contributions.append({
        "point_id": search_result["point_id"],
        "inside_places": len(inside_ids),
        "new_inside_places": len(new_ids),
        "cumulative_inside_places": len(seen_inside_ids),
        "potentially_truncated": search_result["potentially_truncated"]
    })

for contribution in query_contributions:
    print(contribution)

In [0]:
# Summarize primary types and missing ratings within the study area.

from collections import Counter

inside_places = []

for place in places_by_id.values():
    location = place.get("location") or {}
    latitude = location.get("latitude")
    longitude = location.get("longitude")

    if latitude is None or longitude is None:
        continue

    distance = distance_meters(
        study_center["latitude"],
        study_center["longitude"],
        latitude,
        longitude
    )

    if distance <= 600:
        inside_places.append(place)

primary_type_counts = Counter(
    place.get("primaryType") or "NOT_REPORTED"
    for place in inside_places
)

missing_rating = sum(
    place.get("rating") is None
    for place in inside_places
)

missing_review_count = sum(
    place.get("userRatingCount") is None
    for place in inside_places
)

missing_cafe_tag = sum(
    "cafe" not in (place.get("types") or [])
    for place in inside_places
)

print(f"Places inside study area: {len(inside_places)}")

print("\nPrimary types:")
for primary_type, count in primary_type_counts.most_common():
    print(f"  {primary_type}: {count}")

print(f"\nMissing rating: {missing_rating}")
print(f"Missing review count: {missing_review_count}")
print(f"Places without cafe tag: {missing_cafe_tag}")

In [0]:
# Collect restaurant responses across the nine Sopocachi search points.

import time

if "restaurant_search_results" not in globals():
    restaurant_search_results = []

completed_point_ids = {
    item["point_id"]
    for item in restaurant_search_results
}

pending_points = [
    point
    for point in search_points
    if point["point_id"] not in completed_point_ids
]

max_calls = 9

for index, search_point in enumerate(pending_points[:max_calls]):
    if index > 0:
        time.sleep(12)

    try:
        search_result = fetch_nearby_places(
            api_key=google_places_api_key,
            included_types=["restaurant"],
            latitude=search_point["latitude"],
            longitude=search_point["longitude"],
            radius_meters=search_point["radius_meters"]
        )
    except requests.RequestException as error:
        print(
            f"Stopped at {search_point['point_id']} | "
            f"Error type: {type(error).__name__}"
        )
        break

    returned_count = len(
        search_result["response_data"].get("places", [])
    )

    search_result.update({
        "query_id": f"sopocachi_restaurants_{search_point['point_id']}",
        "study_area": "sopocachi",
        "requested_category": "restaurants",
        "point_id": search_point["point_id"],
        "study_center": study_center.copy(),
        "study_radius_meters": 600,
        "returned_count": returned_count,
        "potentially_truncated": returned_count == 20
    })

    restaurant_search_results.append(search_result)

    print(
        f"{search_point['point_id']} | "
        f"Returned: {returned_count} | "
        f"Potentially truncated: {returned_count == 20}"
    )

print(
    f"Restaurant responses registered: "
    f"{len(restaurant_search_results)}"
)

In [0]:
# Summarize restaurant coverage, primary types, and missing fields.

from collections import Counter

restaurant_places_by_id = {}
restaurant_appearances = 0
restaurant_missing_ids = 0

for search_result in restaurant_search_results:
    returned_places = search_result["response_data"].get("places", [])
    restaurant_appearances += len(returned_places)

    for place in returned_places:
        place_id = place.get("id")

        if not place_id:
            restaurant_missing_ids += 1
            continue

        if place_id not in restaurant_places_by_id:
            restaurant_places_by_id[place_id] = place

restaurant_inside_places = []
restaurant_outside_count = 0
restaurant_missing_location = 0

for place in restaurant_places_by_id.values():
    location = place.get("location") or {}
    latitude = location.get("latitude")
    longitude = location.get("longitude")

    if latitude is None or longitude is None:
        restaurant_missing_location += 1
        continue

    distance = distance_meters(
        study_center["latitude"],
        study_center["longitude"],
        latitude,
        longitude
    )

    if distance <= 600:
        restaurant_inside_places.append(place)
    else:
        restaurant_outside_count += 1

print(f"Total appearances: {restaurant_appearances}")
print(f"Unique places: {len(restaurant_places_by_id)}")
print(
    "Repeated appearances:",
    restaurant_appearances
    - restaurant_missing_ids
    - len(restaurant_places_by_id)
)
print(f"Appearances without ID: {restaurant_missing_ids}")
print(f"Inside study area: {len(restaurant_inside_places)}")
print(f"Outside study area: {restaurant_outside_count}")
print(f"Missing location: {restaurant_missing_location}")

print("\nPrimary types:")
for primary_type, count in Counter(
    place.get("primaryType") or "NOT_REPORTED"
    for place in restaurant_inside_places
).most_common():
    print(f"  {primary_type}: {count}")

print(
    "\nMissing rating:",
    sum(place.get("rating") is None for place in restaurant_inside_places)
)
print(
    "Missing review count:",
    sum(
        place.get("userRatingCount") is None
        for place in restaurant_inside_places
    )
)
print(
    "Places without restaurant tag:",
    sum(
        "restaurant" not in (place.get("types") or [])
        for place in restaurant_inside_places
    )
)

In [0]:
# Check rating completeness and overlap between cafés and restaurants.

cafe_ids = {
    place["id"]
    for place in inside_places
    if place.get("id")
}

restaurant_ids = {
    place["id"]
    for place in restaurant_inside_places
    if place.get("id")
}

print(
    "Restaurants missing rating:",
    sum(
        place.get("rating") is None
        for place in restaurant_inside_places
    )
)

print(
    "Restaurants missing review count:",
    sum(
        place.get("userRatingCount") is None
        for place in restaurant_inside_places
    )
)

print(
    "Places without restaurant tag:",
    sum(
        "restaurant" not in (place.get("types") or [])
        for place in restaurant_inside_places
    )
)

print(
    "Places appearing in both categories:",
    len(cafe_ids & restaurant_ids)
)

print(
    "Unique places across both categories:",
    len(cafe_ids | restaurant_ids)
)

In [0]:
# Compare observed review counts for cafés and restaurants.

from statistics import median

category_samples = {
    "cafes": inside_places,
    "restaurants": restaurant_inside_places
}

for category, sample in category_samples.items():
    review_counts = [
        place["userRatingCount"]
        for place in sample
        if place.get("userRatingCount") is not None
    ]

    print({
        "category": category,
        "places_in_sample": len(sample),
        "places_with_review_count": len(review_counts),
        "median_review_count": (
            median(review_counts) if review_counts else None
        ),
        "minimum_review_count": (
            min(review_counts) if review_counts else None
        ),
        "maximum_review_count": (
            max(review_counts) if review_counts else None
        )
    })
    

## Initial exploration — Findings

### Scope
- Area: within 600 meters of Plaza Abaroa, La Paz.
- Categories tested: cafés and restaurants.
- Method: Nearby Search ranked by distance, using a 3 × 3 grid.
- Grid spacing: 300 meters; search radius per grid point: 350 meters.
- Cafés also included an initial query centered on Plaza Abaroa.

### Observed results

| Metric | Cafés | Restaurants |
|---|---:|---:|
| API responses | 10 | 9 |
| Returned appearances | 198 | 180 |
| Unique places within the study area | 111 | 172 |
| Places with a reported review count | 82 | 149 |
| Places missing a review count | 29 | 23 |
| Median reported review count | 11.5 | 11 |
| Minimum reported review count | 1 | 1 |
| Maximum reported review count | 2,852 | 10,634 |
| Responses reaching the 20-result limit | 9 | 9 |

Six places appeared in both categories, giving 277 unique places
within the study area across the two categories.

### Data handling decisions
- Preserve original place types and query categories separately.
- Identify repeated appearances using place_id.
- Keep missing review counts as null, rather than replacing them with zero.
- Separate records outside the study radius from the analytical sample.
- Preserve query parameters and extraction timestamps for traceability.

### Limitations
These are partial API search results, not a census of businesses.
Reaching the result limit indicates potentially incomplete coverage.
The observed medians describe only records with reported review counts.
The results do not yet test the hypothesis against other business categories
or establish demand for an NFC solution.

### Next stage
Prepare reusable extraction configuration, response persistence,
and execution controls before expanding to other categories and areas.

In [0]:
%sql
-- List external locations visible to the current user.
SHOW EXTERNAL LOCATIONS;

In [0]:
# Check that the exploration responses are still available before saving.

expected_variables = [
    "search_results",
    "restaurant_search_results"
]

for variable_name in expected_variables:
    if variable_name in globals():
        print(
            f"{variable_name}: "
            f"{len(globals()[variable_name])} responses available"
        )
    else:
        print(f"{variable_name}: NOT AVAILABLE")

In [0]:
# Define and test immediate response persistence using synthetic data.

import json
from pathlib import Path
from uuid import uuid4
from datetime import datetime, timezone

LANDING_PATH = Path(
    "/Volumes/places_ratings/landing/source_files"
)


def save_extraction(extraction, category, study_area, data_kind):
    extraction_id = uuid4().hex
    saved_at = datetime.now(timezone.utc)

    output_directory = (
        LANDING_PATH
        / data_kind
        / study_area
        / category
        / saved_at.strftime("%Y-%m-%d")
    )
    output_directory.mkdir(parents=True, exist_ok=True)

    output_path = output_directory / f"{extraction_id}.json"

    document = {
        "extraction_id": extraction_id,
        "saved_at_utc": saved_at.isoformat(),
        "data_kind": data_kind,
        "study_area": study_area,
        "requested_category": category,
        "extraction": extraction
    }

    # Exclusive creation prevents overwriting an existing file.
    with output_path.open("x", encoding="utf-8") as file:
        json.dump(document, file, ensure_ascii=False, indent=2)

    return str(output_path)


synthetic_extraction = {
    "extracted_at_utc": datetime.now(timezone.utc).isoformat(),
    "request_parameters": {
        "includedTypes": ["cafe"],
        "radius_meters": 600
    },
    "response_data": {
        "places": [
            {
                "id": "synthetic-place-001",
                "displayName": {"text": "Synthetic Test Café"}
            }
        ]
    }
}

saved_path = save_extraction(
    extraction=synthetic_extraction,
    category="cafes",
    study_area="sopocachi",
    data_kind="synthetic"
)

with open(saved_path, "r", encoding="utf-8") as file:
    recovered_document = json.load(file)

if recovered_document["extraction"] != synthetic_extraction:
    raise ValueError("Persistence check failed: content differs.")

print("Persistence check passed.")
print(f"Saved file: {saved_path}")

In [0]:
# Fetch places and persist the response before returning a summary.

def fetch_and_save_places(
    api_key,
    included_types,
    latitude,
    longitude,
    category,
    study_area,
    radius_meters=600
):
    if not ENABLE_API_CALLS:
        raise RuntimeError(
            "API calls are disabled. Enable them explicitly before extraction."
    )    
    
    extraction = fetch_nearby_places(
        api_key=api_key,
        included_types=included_types,
        latitude=latitude,
        longitude=longitude,
        radius_meters=radius_meters
    )

    saved_path = save_extraction(
        extraction=extraction,
        category=category,
        study_area=study_area,
        data_kind="observed"
    )

    places = extraction["response_data"].get("places", [])

    return {
        "saved_path": saved_path,
        "returned_count": len(places),
        "potentially_truncated": len(places) == 20
    }

In [0]:
# Check that the API function and secret are available in this session.

print(
    "API function available:",
    callable(globals().get("fetch_nearby_places"))
)

google_places_api_key = dbutils.secrets.get(
    scope="places-explorer",
    key="google-places-api-key"
)

print("API key loaded successfully.")

In [0]:
# Define a single-request Google Places extraction without automatic retries.

import requests
from datetime import datetime, timezone


def fetch_nearby_places(
    api_key,
    included_types,
    latitude,
    longitude,
    radius_meters=600
):
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": (
            "places.id,places.displayName,places.primaryType,places.types,"
            "places.formattedAddress,places.location,places.businessStatus,"
            "places.rating,places.userRatingCount"
        )
    }

    payload = {
        "includedTypes": included_types,
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": latitude,
                    "longitude": longitude
                },
                "radius": radius_meters
            }
        }
    }

    extracted_at = datetime.now(timezone.utc).isoformat()

    response = requests.post(
        "https://places.googleapis.com/v1/places:searchNearby",
        headers=headers,
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    return {
        "extracted_at_utc": extracted_at,
        "request_parameters": payload,
        "response_data": response.json()
    }


print("API function defined.")

In [0]:
# Fetch cafés near Plaza Abaroa and immediately save the response to S3.

extraction_summary = fetch_and_save_places(
    api_key=google_places_api_key,
    included_types=["cafe"],
    latitude=-16.5108331,
    longitude=-68.1265597,
    category="cafes",
    study_area="sopocachi",
    radius_meters=600
)

print(extraction_summary)

In [0]:
# Read the saved extraction and verify the reported place count.

with open(
    extraction_summary["saved_path"],
    "r",
    encoding="utf-8"
) as file:
    saved_document = json.load(file)

saved_places = (
    saved_document["extraction"]["response_data"].get("places", [])
)

if len(saved_places) != extraction_summary["returned_count"]:
    raise ValueError("Verification failed: place counts differ.")

print("Saved extraction verified.")
print(f"Places in file: {len(saved_places)}")
print(f"File: {extraction_summary['saved_path']}")

In [0]:
# Build an inventory of persisted API responses without calling Google.

import json
from pathlib import Path

observed_path = Path(
    "/Volumes/places_ratings/landing/source_files/observed"
)

saved_queries = []
unreadable_files = []

for file_path in sorted(observed_path.rglob("*.json")):
    try:
        with file_path.open("r", encoding="utf-8") as file:
            document = json.load(file)

        extraction = document["extraction"]
        parameters = extraction["request_parameters"]
        circle = parameters["locationRestriction"]["circle"]
        returned_places = extraction["response_data"].get("places", [])

        saved_queries.append({
            "file_path": str(file_path),
            "study_area": document["study_area"],
            "category": document["requested_category"],
            "extracted_at_utc": extraction["extracted_at_utc"],
            "latitude": circle["center"]["latitude"],
            "longitude": circle["center"]["longitude"],
            "radius_meters": circle["radius"],
            "request_parameters": parameters,
            "returned_count": len(returned_places),
            "potentially_truncated": len(returned_places) == 20
        })

    except (OSError, ValueError, KeyError, TypeError) as error:
        unreadable_files.append({
            "file_path": str(file_path),
            "error_type": type(error).__name__
        })

print(f"Saved queries found: {len(saved_queries)}")
print(f"Files requiring review: {len(unreadable_files)}")

for query in saved_queries:
    print({
        "area": query["study_area"],
        "category": query["category"],
        "radius_meters": query["radius_meters"],
        "returned_count": query["returned_count"]
    })

In [0]:
# Generate the search grid and identify queries not yet saved in S3.

import math

study_center = {
    "latitude": -16.5108331,
    "longitude": -68.1265597
}

earth_radius_meters = 6_371_000
spacing_meters = 300

search_points = []

for north_offset in [-spacing_meters, 0, spacing_meters]:
    for east_offset in [-spacing_meters, 0, spacing_meters]:
        latitude = study_center["latitude"] + math.degrees(
            north_offset / earth_radius_meters
        )

        longitude = study_center["longitude"] + math.degrees(
            east_offset / (
                earth_radius_meters
                * math.cos(math.radians(study_center["latitude"]))
            )
        )

        search_points.append({
            "point_id": f"sopocachi_{len(search_points) + 1:02d}",
            "latitude": latitude,
            "longitude": longitude,
            "radius_meters": 350
        })


def query_signature(area, category, parameters):
    circle = parameters["locationRestriction"]["circle"]
    center = circle["center"]

    return (
        area,
        category,
        tuple(sorted(parameters.get("includedTypes", []))),
        round(center["latitude"], 7),
        round(center["longitude"], 7),
        circle["radius"],
        parameters.get("rankPreference", "POPULARITY"),
        parameters.get("maxResultCount", 20)
    )


completed_signatures = {
    query_signature(
        query["study_area"],
        query["category"],
        query["request_parameters"]
    )
    for query in saved_queries
}

pending_points = []

for point in search_points:
    parameters = {
        "includedTypes": ["cafe"],
        "rankPreference": "DISTANCE",
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": point["latitude"],
                    "longitude": point["longitude"]
                },
                "radius": point["radius_meters"]
            }
        }
    }

    signature = query_signature("sopocachi", "cafes", parameters)

    if signature not in completed_signatures:
        pending_points.append(point)

print(f"Grid points: {len(search_points)}")
print(f"Pending queries: {len(pending_points)}")

for point in pending_points:
    print(point["point_id"])

In [0]:
# Execute pending café searches, saving every response before continuing.

import time

MAX_CALLS_PER_RUN = 9
calls_attempted = 0
saved_count = 0

ENABLE_API_CALLS = True

try:
    for point in pending_points:
        if calls_attempted >= MAX_CALLS_PER_RUN:
            break

        parameters = {
            "includedTypes": ["cafe"],
            "rankPreference": "DISTANCE",
            "locationRestriction": {
                "circle": {
                    "center": {
                        "latitude": point["latitude"],
                        "longitude": point["longitude"]
                    },
                    "radius": point["radius_meters"]
                }
            }
        }

        signature = query_signature(
            "sopocachi", "cafes", parameters
        )

        if signature in completed_signatures:
            continue

        if calls_attempted > 0:
            time.sleep(12)

        calls_attempted += 1

        try:
            summary = fetch_and_save_places(
                api_key=google_places_api_key,
                included_types=["cafe"],
                latitude=point["latitude"],
                longitude=point["longitude"],
                category="cafes",
                study_area="sopocachi",
                radius_meters=point["radius_meters"]
            )
        except Exception as error:
            print(
                f"Stopped at {point['point_id']} | "
                f"Error type: {type(error).__name__}"
            )
            break

        completed_signatures.add(signature)
        saved_count += 1

        print(
            f"{point['point_id']} | "
            f"Returned: {summary['returned_count']} | "
            f"Potentially truncated: "
            f"{summary['potentially_truncated']} | Saved"
        )

finally:
    ENABLE_API_CALLS = False

print(f"Calls attempted: {calls_attempted}")
print(f"Responses saved: {saved_count}")
print(f"API calls enabled: {ENABLE_API_CALLS}")

In [0]:
# Reload saved Sopocachi café responses and count unique place IDs.

cafe_documents = []
cafe_places_by_id = {}
total_appearances = 0
missing_id_count = 0

for query in saved_queries:
    if (
        query["study_area"] != "sopocachi"
        or query["category"] != "cafes"
    ):
        continue

    with open(query["file_path"], "r", encoding="utf-8") as file:
        document = json.load(file)

    cafe_documents.append(document)

    for place in document["extraction"]["response_data"].get("places", []):
        total_appearances += 1
        place_id = place.get("id")

        if not place_id:
            missing_id_count += 1
            continue

        if place_id not in cafe_places_by_id:
            cafe_places_by_id[place_id] = place

print(f"Files loaded: {len(cafe_documents)}")
print(f"Total appearances: {total_appearances}")
print(f"Unique place IDs: {len(cafe_places_by_id)}")
print(f"Appearances without ID: {missing_id_count}")